<a href="https://colab.research.google.com/github/bsudharsh1979/NATcolab/blob/main/notebooks/0_ALM_Database_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NATcolab** · NVIDIA NeMo Agent Toolkit · *Asset Lifecycle Management with Agentic AI* — Google Colab edition (hosted NVIDIA NIMs)

# Asset Lifecycle Management with Agentic AI - Setup & Introduction

> **🟢 Colab edition.** This is the Google Colab adaptation of the NVIDIA DLI lab notebook. Instead of the lab's Docker containers it uses:
> - **Hosted NVIDIA NIMs** on build.nvidia.com — `nvidia/nemotron-3-ultra-550b-a55b` (planning, tool calling, text-to-SQL), `nvidia/nemotron-3.5-lightning-30b-a3b` (code generation), `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning` (multimodal judge), `nvidia/nemotron-3.5-embed-1b` (embeddings)
> - **MOMENT-1-small**, an open-source time-series foundation model that runs locally, in place of the self-hosted NV-Tesseract NIM
> - a local Python **code sandbox** and a **Phoenix** server started from the notebook
>
> **Before you run:** add your NVIDIA API key (get one at [build.nvidia.com](https://build.nvidia.com)) as a Colab secret named `NVIDIA_API_KEY` — 🔑 icon in the left sidebar → *Add new secret* → enable *Notebook access*. A CPU runtime is enough.
> Colab gives every notebook its own VM, so run the **setup cell below at the top of every notebook** (≈5–8 min on a fresh runtime, seconds afterwards). Paths that were `/dli/task/...` in the lab are `/content/NATcolab/...` here.

In [ ]:
# ⚙️ NATcolab setup — run first (≈5–8 min on a fresh runtime; a few seconds if already done in this session)
# Clones github.com/bsudharsh1979/NATcolab, creates the NAT virtual env (nvidia-nat 1.2.1 + ALM agent),
# builds the NASA turbofan database, trains the RUL model and checks the hosted NVIDIA NIMs.
import os, sys, subprocess
ALM_HOME = os.environ.get("ALM_HOME", "/content/NATcolab")
if not os.path.exists(f"{ALM_HOME}/colab_bootstrap.py"):
    subprocess.run(["git", "clone", "-q", "https://github.com/bsudharsh1979/NATcolab.git", ALM_HOME], check=True)
sys.path.insert(0, ALM_HOME)
os.chdir(ALM_HOME)

import colab_bootstrap
colab_bootstrap.bootstrap()

---
# Course Overview

Welcome to the **Building Observable and Scalable Multi-Agent Workflows for Asset Lifecycle Management** course!

## Learning Objectives

By the end of this course, you will be able to:

**First Half:**
1. Understand and implement Agentic AI workflows using NVIDIA's NeMo Agent Toolkit (NAT)
2. Set up and interact with AI agents for predictive maintenance
3. Visualize and monitor Chain-of-Thought reasoning processes
4. Customize agent behaviors through configuration and tool development
5. Use data retrieval and plotting tools effectively

**Second Half:**
1. Implement observability and monitoring with Phoenix
2. Evaluate agentic workflows using NAT's custom evaluator
3. Build complex queries requiring code generation and execution
4. Understand time-series analysis for asset management using foundation models
5. Apply Time series foundational models for anomaly detection

## Course Structure

This course is divided into 8 notebooks:

- **Notebook 0** (Current): Setup & Introduction
- **Notebook 1**: Basic Interaction & Frontend
- **Notebook 2**: Configuration & Tool Customization
- **Notebook 3**: Observability & Monitoring
- **Notebook 4**: Evaluation & Quality Assurance
- **Notebook 5**: Time-Series Modeling & Foundation Models
- **Optional Notebook**: Code Generation & Complex Queries
- **Assessment**: Final Assessment

## Prerequisites

This course assumes:
- **Python**: Writing classes, function arguments, decorators, and working with libraries like pandas
- **Jupyter Lab**: Navigating the file browser, opening terminals, running cells
- **Linux shell**: Basic commands (`cd`, `cp`, `cat`, editing files)

## What You'll Build

You will build and customize an intelligent Asset Lifecycle Management system that can:
- Query turbofan engine sensor data using natural language
- Predict Remaining Useful Life (RUL) of assets
- Detect anomalies using foundation models
- Generate actionable maintenance recommendations
- Create visualizations and reports automatically
- Execute complex data analysis through code generation

---
# Introduction to Asset Lifecycle Management

## What is Asset Lifecycle Management?

**Asset Lifecycle Management (ALM)** is a comprehensive approach to managing physical assets throughout their entire lifespan, from acquisition to disposal. In industrial settings, effective ALM is critical for:

- **Maximizing asset uptime** and availability
- **Reducing maintenance costs** through predictive strategies
- **Extending asset lifespan** through optimal maintenance scheduling
- **Preventing catastrophic failures** that can cause safety issues and revenue loss
- **Optimizing capital expenditure** by replacing assets at the right time

## The Challenge: Predictive Maintenance

Traditional maintenance strategies include:

1. **Reactive Maintenance**: Fix when broken (high downtime, safety risks)
2. **Preventive Maintenance**: Fixed schedule maintenance (inefficient, over-maintenance)
3. **Predictive Maintenance**: Data-driven maintenance based on actual asset condition (optimal)

**Predictive Maintenance** uses sensor data, machine learning, and analytics to:
- Monitor asset health in real-time
- Predict when maintenance is needed
- Schedule maintenance proactively before failures occur
- Reduce unplanned downtime by up to 50%
- Lower maintenance costs by 10-40%

## Why Agentic AI for Asset Management?

Traditional predictive maintenance systems require:
- Manual data extraction and analysis
- Expert knowledge to interpret sensor patterns
- Multiple tools and dashboards
- Complex workflows for investigations

**Agentic AI** transforms this by:
- Enabling **natural language** queries: "What engines are at risk of failure?"
- **Automatically orchestrating** data retrieval, analysis, and visualization
- **Reasoning through** complex multi-step investigations
- **Generating code** on-the-fly for custom analyses
- **Learning from** domain knowledge embedded in prompts and tools

### Agentic AI Workflow Concepts

An **Agentic AI system** consists of:

1. **Large Language Model (LLM)**: The "brain" that understands queries and plans actions
2. **Tools**: Functions the agent can call (database queries, predictions, visualizations)
3. **Reasoning Agent**: Plans multi-step workflows to answer queries
4. **Execution Agent**: Carries out the plan using available tools
5. **Memory/Context**: Tracks intermediate results and conversation history

**Workflow Example:**
```
User Query: "Show me engines with unusual sensor readings and predict their failure risk"

Agent Reasoning:
1. Retrieve sensor data for all engines → SQL Tool
2. Detect anomalies in sensor readings → Anomaly Detection Tool
3. For anomalous engines, predict RUL → RUL Prediction Tool
4. Visualize results → Plotting Tool
5. Generate maintenance recommendations → Recommendation Tool
6. Return comprehensive answer with visualizations
```

---
# NASA Turbofan Engine Dataset

## Dataset Overview

In this course, we use the **NASA Commercial Modular Aero-Propulsion System Simulation (C-MAPSS)** dataset, which simulates turbofan engine degradation.

**Source:** [NASA Prognostics Data Repository](https://www.kaggle.com/datasets/behrad3d/nasa-cmaps)

## Dataset Structure

The dataset contains 4 subsets (FD001-FD004), each with different operating conditions and fault modes:

| Dataset | Operating Conditions | Fault Modes | Training Engines | Test Engines |
|---------|---------------------|-------------|------------------|-------------|
| FD001   | 1 (Sea Level)       | 1 (HPC)     | 100              | 100         |
| FD002   | 6 (Various)         | 1 (HPC)     | 260              | 259         |
| FD003   | 1 (Sea Level)       | 2 (HPC+Fan) | 100              | 100         |
| FD004   | 6 (Various)         | 2 (HPC+Fan) | 249              | 248         |

## Data Columns

Each row represents one operational cycle for an engine:

1. **unit_number**: Engine identifier (1-N)
2. **time_in_cycles**: Operational cycle number (starts at 1)
3. **operational_setting_1, _2, _3**: Flight conditions (altitude, throttle, etc.)
4. **sensor_measurement_1** through **sensor_measurement_21**: Sensor readings
5. **RUL** (in train set only): Remaining Useful Life - cycles until failure

## Sensor Types

The 21 sensors measure various aspects of engine health:
- Temperature measurements (fan, core, turbine)
- Pressure measurements (fan, bypass, core, turbine)
- Physical sensor readings (fan/core speeds, fuel flow, etc.)
- Static pressure measurements

## Engine Lifecycle Pattern

```
Cycle 1 ──────────────────────────────────► Cycle N (Failure)
│                                             │
│  Normal Operation                           │  Degraded
│  ▪ Sensors show baseline values             │  ▪ Sensors drift
│  ▪ No anomalies                             │  ▪ Anomalies appear
│                                             │  ▪ RUL = 0
│                                             │
└─────────── Progressive Degradation ─────────┘
           RUL decreases with each cycle
```

## Predictive Maintenance Goal

Using this data, we want to:
1. **Monitor**: Track sensor patterns over time
2. **Detect**: Identify abnormal sensor readings (anomalies)
3. **Predict**: Estimate Remaining Useful Life (RUL)
4. **Recommend**: Suggest maintenance actions before failure

## Database Setup

The dataset has been preprocessed and loaded into a **SQLite database** with the following structure:

```sql
Tables:
- training_data  (all training data with 'dataset' column: FD001, FD002, FD003, FD004)
- test_data      (all test data with 'dataset' column)
- rul_data       (ground truth RUL values)
- sensor_metadata (descriptions of each sensor)
- dataset_metadata (information about each FD dataset)

Schema (all tables):
- unit_number (INTEGER)
- time_in_cycles (INTEGER)
- operational_setting_1, _2, _3 (REAL)
- sensor_measurement_1 through _21 (REAL)
- RUL (INTEGER, in train tables only)
```

---
# Section 1: Environment Setup

The setup cell at the top of this notebook already prepared everything. Let's verify it.

> **Note:** In Colab the course files live in `/content/NATcolab/` (open the 📁 file browser on the left). All paths in this course are relative to this directory unless stated otherwise.

## 1.1 Validate Virtual Environment

The NAT backend runs in its own Python 3.12 virtual environment (`/content/alm-venv`, the equivalent of the lab's `/opt/predm-lab`), separate from the notebook kernel. Execute the following code block to confirm both are in place:

In [ ]:
import sys, os, shutil, subprocess

venv_python = str(colab_bootstrap.VENV_PY)
print(f"Notebook kernel : {sys.executable} (Python {sys.version.split()[0]})")
print(f"NAT venv        : {os.environ['ALM_VENV']}")
print("NAT venv Python :", subprocess.run([venv_python, "--version"], capture_output=True, text=True).stdout.strip())
print("nvidia-nat      :", subprocess.run([venv_python, "-c", "import importlib.metadata as m; print(m.version('nvidia-nat'))"],
                                          capture_output=True, text=True).stdout.strip())
print(f"UV: {'✅' if shutil.which('uv') else '❌'}")

## 1.2 Setup Database

The NASA Turbofan dataset is located at: [NASA C-MAPSS on Kaggle](https://www.kaggle.com/datasets/behrad3d/nasa-cmaps)

The setup cell already downloaded the raw data and built the database. Run the setup script again to see what it does (it is idempotent — existing files are reused):

In [ ]:
!{colab_bootstrap.VENV_PY} setup_database.py

## 1.3 Explore the Dataset

Let's take a quick look at the data structure:

In [ ]:
import sqlite3
import pandas as pd
from IPython.display import display

db_path = "database/nasa_turbo.db"
conn = sqlite3.connect(db_path)

# ── Available tables ───────────────────────────────────────────────────────────
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Available tables:")
print(tables.to_string(index=False))

# ── Sample data: train_FD001 (unit 1, first 5 cycles) ─────────────────────────
print("\nSample data — train_FD001 (unit 1, first 5 cycles):")
sample = pd.read_sql_query(
    "SELECT * FROM train_FD001 WHERE unit_number=1 LIMIT 5", conn
)
display(sample.style.format("{:.4f}", subset=sample.select_dtypes("float").columns).hide(axis="index"))

# ── Dataset statistics: FD001 ──────────────────────────────────────────────────
print("\nDataset statistics — FD001:")
stats = pd.read_sql_query(
    """SELECT
        COUNT(DISTINCT unit_number) AS num_engines,
        MAX(time_in_cycles)         AS max_cycles,
        MIN(RUL)                    AS min_rul,
        MAX(RUL)                    AS max_rul
       FROM train_FD001""",
    conn,
)
display(stats.style.hide(axis="index"))

conn.close()

## 1.4 RUL Prediction Model

The `predict_rul` tool uses an XGBoost model (`models/xgb_model_fd001.pkl` + `models/scaler_model.pkl`). The DLI image ships it pre-trained; in Colab the setup cell trains it with `train_rul_model.py` on the `train_FD001` table (12 sensor features, RUL capped at 125 cycles):

In [ ]:
!{colab_bootstrap.VENV_PY} train_rul_model.py
for f in sorted(os.listdir("models")):
    print(f"models/{f:<22} {os.path.getsize(os.path.join('models', f)):>10,} bytes")

## Verify Backend Services

In the DLI lab, several containers run behind the scenes (a local **NIM** for the LLMs and **NV-Tesseract** for time-series anomaly detection). In this Colab edition the LLMs and the embedding model are **hosted NVIDIA NIMs** called over HTTPS, and anomaly detection uses the local **MOMENT** model. Run the cell below to confirm the hosted NIM endpoints answer with your key:

In [ ]:
import colab_bootstrap
assert colab_bootstrap.check_nims(), "Some NIM endpoints did not answer — check your NVIDIA_API_KEY / quota"

---
# Next Steps

Congratulations! You've completed the environment setup and learned about:
- Asset Lifecycle Management and predictive maintenance
- Agentic AI concepts and workflow patterns
- NASA Turbofan dataset structure

You're now ready to move on to **Notebook 1** where you'll:
- Learn about NVIDIA NeMo Agent Toolkit (NAT) architecture
- Install the NAT agent package and its dependencies
- Start the NAT backend service
- Send your first queries to the agent
- Set up the frontend UI
- Monitor agent reasoning in real-time

**[Continue to Notebook 1 - Basic Interaction →](1_ALM_Application_Setup.ipynb)**